## Sentimental Analysis --Spark-nlp  part2


In [1]:
# pip install pyprind -i https://pypi.tuna.tsinghua.edu.cn/simple
import pyprind
import pandas as pd
import os
import numpy as np

In [2]:
#import nltk
#nltk.download('stopwords')

# or manually download stopword from https://www.nltk.org/nltk_data/
# and save it in your disk drive
# Set your NLTK_DATA environment variable to point to your top level nltk_data folder.
# for example, your nltk_data folder is C:/nltk_data
# you can save stopword under C:/nltk_data folder

In [3]:
# preprocessing, remove stop words
from nltk.corpus import stopwords
import re
stop=stopwords.words('english')
def tokenizer(text):
    text=re.sub('<[^>]*>','',text)
    emoticons=re.findall('(?::|;|=)(?:-)?(?:</span>|<spanclass="es0">|D|P)',text.lower())
    text=re.sub('[\W]+',' ',text.lower())+' '.join(emoticons).replace('-','')
    tokenized=[w for w in text.split() if w not in stop]
    return tokenized

In [4]:
def stream_docs(path):
    with open(path,'r', encoding='UTF-8') as csv:
        next(csv)# skip header
        for line in csv:
            text,label=line[:-3],int(line[-2])
            yield text,label

In [5]:
def get_minibatch(doc_stream,size):
    docs,y=[],[]
    try:
        for _ in range(size):
            text,label=next(doc_stream)
            docs.append(text)
            y.append(label)
    except StopIteration:
            return None,None
    return docs,y

## sklearn.feature_extraction.text.HashingVectorizer

class sklearn.feature_extraction.text.HashingVectorizer(*, input='content', encoding='utf-8', decode_error='strict', strip_accents=None, lowercase=True, preprocessor=None, tokenizer=None, stop_words=None, token_pattern='(?u)\\b\\w\\w+\\b', ngram_range=(1, 1), analyzer='word', n_features=1048576, binary=False, norm='l2', alternate_sign=True, dtype=<class 'numpy.float64'>)

Convert a collection of text documents to a matrix of token occurrences.

It turns a collection of text documents into a scipy.sparse matrix holding token occurrence counts (or binary occurrence information), possibly normalized as token frequencies if norm=’l1’ or projected on the euclidean unit sphere if norm=’l2’.

This text vectorizer implementation uses the hashing trick to find the token string name to feature integer index mapping.

This strategy has several advantages:

it is very low memory scalable to large datasets as there is no need to store a vocabulary dictionary in memory.

it is fast to pickle and un-pickle as it holds no state besides the constructor parameters.

it can be used in a streaming (partial fit) or parallel pipeline as there is no state computed during fit.

There are also a couple of cons (vs using a CountVectorizer with an in-memory vocabulary):

there is no way to compute the inverse transform (from feature indices to string feature names) which can be a problem when trying to introspect which features are most important to a model.

there can be collisions: distinct tokens can be mapped to the same feature index. However in practice this is rarely an issue if n_features is large enough (e.g. 2 ** 18 for text classification problems).

no IDF weighting as this would render the transformer stateful.


## sklearn.linear_model.SGDClassifier

class sklearn.linear_model.SGDClassifier(loss='hinge', *, penalty='l2', alpha=0.0001, l1_ratio=0.15, fit_intercept=True, max_iter=1000, tol=0.001, shuffle=True, verbose=0, epsilon=0.1, n_jobs=None, random_state=None, learning_rate='optimal', eta0=0.0, power_t=0.5, early_stopping=False, validation_fraction=0.1, n_iter_no_change=5, class_weight=None, warm_start=False, average=False)

Linear classifiers (SVM, logistic regression, etc.) with SGD training.

This estimator implements regularized linear models with stochastic gradient descent (SGD) learning: the gradient of the loss is estimated each sample at a time and the model is updated along the way with a decreasing strength schedule (aka learning rate). SGD allows minibatch (online/out-of-core) learning via the partial_fit method. For best results using the default learning rate schedule, the data should have zero mean and unit variance.

This implementation works with data represented as dense or sparse arrays of floating point values for the features. The model it fits can be controlled with the loss parameter; **by default, it fits a linear support vector machine (SVM)**.

The regularizer is a penalty added to the loss function that shrinks model parameters towards the zero vector using either the squared euclidean norm L2 or the absolute norm L1 or a combination of both (Elastic Net). If the parameter update crosses the 0.0 value because of the regularizer, the update is truncated to 0.0 to allow for learning sparse models and achieve online feature selection.

In [6]:
# feature extraction and vectorization
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier

vect = HashingVectorizer(decode_error='ignore',n_features=2**21,preprocessor=None,tokenizer=tokenizer)

clf = SGDClassifier(loss='log',random_state=1,n_iter_no_change=1)

In [7]:
# train the model
import pyprind
pbar=pyprind.ProgBar(40)
classes=np.array([0,1])

doc_stream_train = stream_docs(path='./data/movie_train.csv')

for _ in range(40):
    x_train,y_train=get_minibatch(doc_stream_train,size=1000)
    if not x_train:
        break
        
    x_train=vect.transform(x_train) # word to vec, also called word embedding
    
    clf.partial_fit(x_train,y_train,classes=classes)
    
    pbar.update()

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:24


In [8]:
#evaluate the model
doc_stream_test = stream_docs(path='./data/movie_test.csv')

x_test,y_test=get_minibatch(doc_stream_test,size=10000)
x_test=vect.transform(x_test)

print('accuracy: %.3f' % clf.score(x_test,y_test))

accuracy: 0.869
